In [ ]:
# Import Packages
from __future__ import annotations
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import pymap3d as pm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from IPython.display import display, HTML

import os
import sys
import imageio.v2 as imageio
import glob
import open3d as o3d
import healpix as hp

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from geopy.distance import great_circle
from scipy.interpolate import CubicSpline

from matplotlib.patches import Polygon
from astropy_healpix import HEALPix
from matplotlib.colors import ListedColormap, BoundaryNorm, to_rgba
from matplotlib.cm import ScalarMappable

import gzip
import shutil
import os
from pathlib import Path

from urllib.request import urlopen, urlretrieve
from urllib.error import HTTPError, URLError
import tempfile
import plotly.graph_objects as go
from shapely.geometry import Point
from shapely.strtree import STRtree
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable

import cartopy.feature as cfeature
from shapely.geometry import LineString, MultiLineString

from bokeh.plotting import figure, show, output_notebook
from bokeh.plotting import figure, show
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource,
    Slider,
    Select,
    CustomJS,
    Div,
    CheckboxGroup,
    HoverTool,
    Legend,
    LegendItem, Button, WheelZoomTool, PanTool, ResetTool, SaveTool
)
from bokeh.layouts import column

from bokeh.io import output_notebook, show

output_notebook()

%reload_ext autoreload
%autoreload 2


In [ ]:
import sys
from pathlib import Path

# Project structure assumed:
# project/
# ├── notebooks/
# └── Functions/

PROJECT_ROOT = Path.cwd().parent
FUNCTIONS_DIR = PROJECT_ROOT / "functions"

if not FUNCTIONS_DIR.exists():
    raise FileNotFoundError(f"Functions directory not found: {FUNCTIONS_DIR}")

if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))

print("Using functions directory:", FUNCTIONS_DIR)


In [ ]:
from collective0 import build_ipp_validation_df, ipp_pipeline_one_station, ipp_pipeline_multiple_stations, generate_ismr_paths_by_station
from ipp_on_map import plot_ipp_difference_magnitude, plot_ipp_map_geodetic, plot_ipp_sp3, plot_ipp_sp3_colored_by_time, make_ipp_sp3_gif, compute_geodetic_differences_ipp
from ToD_grid import define_laea_projection, build_laea_solution2_hierarchy
from AIMS_IT_value import (add_dTEC_dt_abs,compute_I_T_AIMS,add_I_T_color_AIMS,
    compute_full_I_T_pipeline_AIMS,
    plot_AIMS_IT_per_cell_at_time,
    plot_AIMS_IT_per_cell_at_time_with_colored_IPPs,
    classify_IT_value,
    IT_class_to_color,
    plot_laea_solution2_IT_timeline_multilevel_bokeh,
    compute_coverage_quality_by_level,
    plot_laea_solution2_IT_timeline_multilevel_bokeh_coverage_quality, 
    collect_ismr_paths_from_extracted_folder, 
    assign_ipp_to_solution2_hierarchy
    )
laea_crs, wgs84, transformer_wgs84_to_laea, transformer_laea_to_wgs84, map_projection, lat_label_lon = define_laea_projection(projection="3574")


In [ ]:
from pathlib import Path

# --------------------------------------------------
# Data paths
# --------------------------------------------------
# Expected local data structure:
# Sarah Schultz Beeck - Mie Bachelor/
# ├── Data/
# │   ├── All_stations.csv
# │   ├── COD0OPSFIN_20242840000_01D_05M_ORB.SP3
# │   └── ISMR/
# │       └── 24284/
# └── mie_bachelor_code/
#     └── notebooks/
#         └── your_notebook.ipynb

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT.parent / "Data"

path_stations_SWADO = DATA_DIR / "All_stations.csv"
path_sp3 = DATA_DIR / "COD0OPSFIN_20242840000_01D_05M_ORB.SP3"
base_dir = DATA_DIR / "ISMR" / "24284"

print("Data directory:", DATA_DIR)
print("Station file exists:", path_stations_SWADO.exists())
print("SP3 file exists:", path_sp3.exists())
print("ISMR folder exists:", base_dir.exists())

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT.parent / "Data"

ismr_root = DATA_DIR / "ISMR" / "24284"

paths_all_stations_SWADO = collect_ismr_paths_from_extracted_folder(
    root_folder=ismr_root,
    extracted_folder_name="extracted_gz",
    file_extension=".ismr"
)

print("ISMR root:", ismr_root)
print("ISMR root exists:", ismr_root.exists())
print("Stations found:", list(paths_all_stations_SWADO.keys()))

from pathlib import Path

# --------------------------------------------------
# Data paths
# --------------------------------------------------
# Expected local structure:
# Sarah Schultz Beeck - Mie Bachelor/
# ├── Data/
# │   ├── CHAIN_stations.csv
# │   ├── COD0OPSFIN_20242840000_01D_05M_ORB.SP3
# │   └── ISMR/
# │       └── CHAIN/
# │           └── *.ismr
# └── mie_bachelor_code/
#     └── notebooks/
#         └── your_notebook.ipynb

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT.parent / "Data"

path_stations_CHAIN = DATA_DIR / "CHAIN_stations.csv"
path_sp3 = DATA_DIR / "COD0OPSFIN_20242840000_01D_05M_ORB.SP3"
base_dir = DATA_DIR / "ISMR"

spline = True

# --------------------------------------------------
# Collect CHAIN ISMR files
# --------------------------------------------------
paths_all_stations = collect_ismr_paths_from_extracted_folder(
    root_folder=base_dir,
    extracted_folder_name="CHAIN",
    file_extension=".ismr",
    station_id_length=4
)

paths_all_stations = {
    k.upper(): v
    for k, v in paths_all_stations.items()
}

stations_to_run = sorted(paths_all_stations.keys())

print("Data directory:", DATA_DIR)
print("CHAIN station file exists:", path_stations_CHAIN.exists())
print("SP3 file exists:", path_sp3.exists())
print("ISMR base folder exists:", base_dir.exists())

print("\nStations found in ISMR files:")
print(stations_to_run)



In [ ]:
elevation_mask = 15
h_ion = 350_000.0
R_E = 6_378_137.0
one_station = "SWAF"
stations_to_run_SWADO = ["SNOR", "THU5", "SWAF", "KLQ2", "KSUK", "SCO3", "UPV1","QAQ3"]
spline = True #True/False

# Import SWADO data

In [ ]:
df_val_SWADO, station_info_SWADO = ipp_pipeline_multiple_stations(
    elevation_mask=elevation_mask,
    R_E=R_E,
    h_ion=h_ion,
    path_stations=path_stations_SWADO,
    stations_to_run=stations_to_run_SWADO,
    paths_ismr_by_station=paths_all_stations_SWADO,
    path_sp3=path_sp3,
    use_spline=spline,
)

In [ ]:
df_SWADO_assigned, grid_large, grid_medium, grid_small = assign_ipp_to_solution2_hierarchy(
    df_points=df_val_SWADO,
    cell_size=1_200_000,
    lat_south=44,
    transformer_wgs84_to_laea=transformer_wgs84_to_laea,
    lon_col="ipp_sp3_lon",
    lat_col="ipp_sp3_lat"
)

# IMPORT CHAIN data

In [ ]:
df_val_CHAIN, station_info = ipp_pipeline_multiple_stations(
    elevation_mask=elevation_mask,
    R_E=R_E,
    h_ion=h_ion,
    path_stations=path_stations_CHAIN,
    stations_to_run=stations_to_run,
    paths_ismr_by_station=paths_all_stations,
    path_sp3=path_sp3,
    use_spline=spline,
)

In [ ]:
df_CHAIN_assigned, grid_large, grid_medium, grid_small = assign_ipp_to_solution2_hierarchy(
    df_points=df_val_CHAIN,
    cell_size=1_200_000,
    lat_south=44,
    transformer_wgs84_to_laea=transformer_wgs84_to_laea,
    lon_col="ipp_sp3_lon",
    lat_col="ipp_sp3_lat"
)

# CHAIN + SWADO

In [ ]:
df_SWADO_assigned_all = df_SWADO_assigned.copy()
df_CHAIN_assigned_all = df_CHAIN_assigned.copy()

df_SWADO_assigned_all["network"] = "SWADO"
df_CHAIN_assigned_all["network"] = "CHAIN"

df_all_assigned = pd.concat(
    [df_SWADO_assigned_all, df_CHAIN_assigned_all],
    ignore_index=True)


In [ ]:
coverage_df = compute_coverage_quality_by_level(
    df_all_assigned=df_all_assigned,
    time_col="UTC Time",
    time_freq="min",
    station_col="Station",
    min_ipp_per_cell=4,
    n_good_large=12,
    n_good_medium=8,
    n_good_small=4,
    receiver_good=3,
    use_satellite=True
)

In [ ]:
print("Columns in df_all_assigned:")
print(df_all_assigned.columns.tolist())

possible_it_cols = [c for c in df_all_assigned.columns if "I" in c or "T" in c or "it" in c.lower()]
print("Possible IT-related columns:")
print(possible_it_cols)

In [ ]:
# Make sure UTC Time is datetime
df_all_assigned["UTC Time"] = pd.to_datetime(df_all_assigned["UTC Time"])

# Compute dTEC_dt_abs, I_T and I_T_color
df_all_assigned = compute_full_I_T_pipeline_AIMS(df_all_assigned)

# Check that I_T now exists
print("I_T" in df_all_assigned.columns)
print(df_all_assigned[["S4", "Phi60", "TEC", "dTEC_dt_abs", "I_T"]].head())

In [ ]:
 layout, p, slider, grid_select, statistic_select, grid_frames_by_level, ipp_frames_by_level, df_plot = plot_laea_solution2_IT_timeline_multilevel_bokeh_coverage_quality(
     df_ipp=df_all_assigned,
     coverage_df=coverage_df,
     cell_size=1_200_000,
     lat_south=50,
     projection="3574",
     initial_grid_level="medium",
     time_col="UTC Time",
     time_freq="min",
     time_start="2024-10-10 19:00:00",
     time_end="2024-10-10 23:59:00",
     min_ipp_per_cell=4,
     it_value_col="I_T",
     csv_path_swado=path_stations_SWADO,
     csv_path_chain=path_stations_CHAIN)

 show(layout)